# 24 · Laser linewidth from a delayed self-heterodyne measurement

**Goal**: characterize the phase noise of a CW laser - its FM-noise PSD and
its (Lorentzian) linewidth - from a *digitized interferometric beat*, using
`commkit.analysis.dsh_phase`, `dsh_fm_noise_psd`, and `estimate_linewidth`
with the `DshFmPsd`, `DshIncrement` and `DshLorentzian` methods.

## The physical setup

A laser's absolute phase noise cannot be observed directly: you would need a
second, much better laser to beat against.  The delayed self-interference
trick manufactures a reference **out of the laser itself**:

```
                    +-------- fiber delay τ_d --------+
  laser --> 50/50 --+                                 +--> 50/50 --> PD --> ADC
                    +-- AOM (shift f_shift) ----------+
```

* One arm is delayed by `τ_d = n·L/c` (≈ 4.9 µs per km of SMF).
* The other arm is frequency-shifted by an AOM at `f_shift` (typically
  40-200 MHz) so the beat lands away from DC - **self-heterodyne**: a single
  photodetector suffices and the ADC records *real* samples.
* The photodetector sees the interference of the laser with a delayed copy
  of itself; the beat carries the **differential phase**

  $$\Delta\varphi(t) = \varphi(t) - \varphi(t-\tau_d).$$

This notebook covers this classic **AOM + single-photodetector receiver**
(real samples).  The receiver is an independent choice, though: the same
AOM setup read out with a coherent (90°-hybrid IQ) receiver delivers
*complex* samples that every function below accepts directly - the Hilbert
step and its Nyquist-margin constraint (checklist item 3) simply drop out.
The AOM-free variant - the hybrid with `f_shift = 0` (self-homodyne) - has
its own walkthrough in [26](26-laser-linewidth-self-homodyne-iq.ipynb).

Whether the differential phase has "decohered" is governed by the ratio of
the delay to the **coherence time** `τ_c = 1/(π·Δν)`:

| regime | condition | beat spectrum |
|---|---|---|
| **incoherent** | `τ_d ≫ τ_c` | smooth Lorentzian of FWHM `2Δν` |
| **coherent** | `τ_d ≲ τ_c` | carrier spike + fringes spaced `1/τ_d` |

The **main path of this notebook is the classic long-delay (incoherent)
measurement** - for the 100 kHz-1 MHz-class lasers of coherent transmission,
`τ_d ≫ τ_c` costs only a few tens of km of fiber, and all three estimators
apply.  The coherent short-delay regime, the *necessary* choice for
kHz-class lasers where the decoherence spool would grow to hundreds of km,
is the subject of [25](25-laser-linewidth-short-delay-and-flicker.ipynb), together with flicker noise,
the Allan deviation and polarization fading.

The laser model (white and flicker FM noise, the Lorentzian line) is
introduced in [09](09-laser-phase-noise.ipynb).

## 0. Backend and reproducibility

CommKit dispatches on the array type: build the arrays with CuPy and
every `analysis` call below runs on the GPU; build them with NumPy and it
runs on the CPU.  Only hand-made matplotlib overlays need an explicit
device->host transfer (`to_device(x, "cpu")`) - the library plot functions
handle that internally.

All randomness flows through *seeded library generators*
(`generate_phase_noise`, `apply_awgn`) - no module-level RNG, so every cell
is individually re-runnable with identical results.  The noise is drawn on
`DEVICE`: CPU and GPU runs see different realizations with the same
statistics, so the estimates agree to within their scatter.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from commkit import analysis, plotting
from commkit.backend import is_cupy_available, to_device
from commkit.impairments import apply_awgn, generate_phase_noise

if is_cupy_available():
    import cupy as xp  # arrays (and all analysis) on the GPU
else:
    xp = np  # CPU fallback - everything below is backend-agnostic


def cpu(a):
    """Device->host helper for hand-made matplotlib overlays."""
    return to_device(a, "cpu")


DEVICE = "cpu" if xp is np else "gpu"

plotting.apply_default_theme()

## 1. Simulate the measurement - step by step

A 200 kHz white-FM laser (`τ_c = 1.6 µs`) against a **25 km spool**
(`τ_d = 122.5 µs`, coherence factor `π·Δν·τ_d ≈ 77` -> safely incoherent).

**Step 1 - the laser.**  The canonical laser phase-noise model is **white
FM noise**: the instantaneous frequency is white, so the phase is a Wiener
process (random walk) and the field spectrum is Lorentzian with FWHM `Δν`:

$$\varphi[n] = \sum_k \delta_k,\qquad \delta_k \sim
\mathcal{N}\!\left(0,\; \tfrac{2\pi\,\Delta\nu}{f_s}\right).$$

`impairments.generate_phase_noise` implements exactly this (its
`linewidth` parameter is `Δν`) and returns the trajectory itself, keeping
the ground truth in hand for every comparison below.

**Step 2 - the interferometer.**  `analysis.dsh_beat` is the forward model
the estimators invert: delay by `τ_d` (rounded to `m = τ_d·f_s` samples),
beat the two copies, modulate onto the AOM carrier -

$$z[n] = \exp\!\big(j\,(2\pi f_\mathrm{shift}\, n/f_s + \varphi[n] -
\varphi[n-m])\big),$$

returning both the beat and the true differential phase `Δφ`.

**Step 3 - detection noise** `w[n]` (shot/thermal), added with
`impairments.apply_awgn` (at 1 sample/symbol, `esn0_db` *is* the beat SNR
in the full Nyquist band).

In [ ]:
FS = 500e6  # ADC sampling rate [Hz]
N = 1 << 21  # record length (≈ 4.2 ms)
DNU_TRUE = 200e3  # laser linewidth to be recovered [Hz]
F_AOM = 80e6  # AOM shift [Hz]
SNR_DB = 25  # beat SNR in the full Nyquist band

M_DELAY = int(round(122.5e-6 * FS))  # 25 km SMF
TAU_D = M_DELAY / FS

phi = generate_phase_noise(
    num_samples=N + M_DELAY, sampling_rate=FS, linewidth=DNU_TRUE, rng=42, device=DEVICE
)
z_dsh, dphi_true = analysis.dsh_beat(phi, sampling_rate=FS, delay=TAU_D, f_shift=F_AOM)
z_dsh = apply_awgn(z_dsh, sps=1, esn0_db=SNR_DB, rng=1)

print(
    f"τ_d = {TAU_D * 1e6:.1f} µs,  τ_c = 1/(πΔν) = {1e6 / (np.pi * DNU_TRUE):.2f} µs, "
    f"τ_d/τ_c = {np.pi * DNU_TRUE * TAU_D:.0f}  (≫ 6 => incoherent regime)"
)

## 2. Look at the beat spectrum first

Always plot the raw beat PSD before trusting any number.  In the incoherent
regime the line at `f_shift` is the **self-convolution of the laser line**:
a smooth Lorentzian of FWHM `2Δν`, no carrier spike, no fringes.  (Spike +
fringes would mean the delay is too short for the classic width recipe -
see [25](25-laser-linewidth-short-delay-and-flicker.ipynb).)

In [ ]:
plotting.plot_psd(
    z_dsh,
    sampling_rate=FS,
    nperseg=1 << 15,
    xlim=((F_AOM - 3e6) / 1e6, (F_AOM + 3e6) / 1e6),
    title=f"Incoherent-regime beat: Lorentzian, FWHM = 2Δν = {2 * DNU_TRUE / 1e3:.0f} kHz",
)
plt.show()

## 3. Extract the differential phase

`dsh_phase` forms the analytic signal (Hilbert transform if the input is
real), removes the beat carrier, and unwraps in float64.  If `f_shift` is
not given, the mean beat frequency is estimated (coarse Kay estimate +
exact least-squares slope removal) - that also *linearly detrends* Δφ,
absorbing the mean laser drift into the returned `f_shift_hz`.  Pass the
known AOM frequency when you want drift to stay visible in Δφ.

The extracted Δφ **is not** the laser phase φ - it is φ filtered by the
interferometer response `H(f) = 1 - e^{-j2πfτ_d}`.  Every estimator below
has to undo (or sidestep) that response.

Sanity check: in the incoherent regime `Var[Δφ] = 2πΔν·τ_d`.

In [ ]:
dphi_est, f_beat = analysis.dsh_phase(z_dsh, sampling_rate=FS, f_shift=F_AOM)
print(f"removed carrier: {f_beat / 1e6:.3f} MHz")
print(
    f"Var[Δφ] measured = {float(xp.var(dphi_est)):.1f} rad², "
    f"theory 2πΔν·τ_d = {2.0 * np.pi * DNU_TRUE * TAU_D:.1f} rad²"
)

## 4. Method 1 - FM-noise PSD fit

**What the interferometer does to the FM noise.**  The beat's instantaneous
frequency is the laser frequency minus its own delayed copy,
`f_beat(t) = f(t) - f(t - τ_d)`.  That is a linear filter, so the laser
FM-noise PSD reaches the beat multiplied by

$$|1 - e^{-j2\pi f\tau_d}|^2 = 4\sin^2(\pi f \tau_d) = 2 - 2\cos(2\pi f\tau_d),$$

a comb of fringes with notches at `f = k/τ_d`.

**What a Welch estimate sees.**  A Welch bin is not a point sample of that
comb: it averages the spectrum over the window's spectral kernel.  Averaging
`cos 2πfτ_d` that way multiplies it by the window's normalized
autocorrelation `ρ_w` at a lag of `τ_d`:

$$E[S_\mathrm{beat}(f)] = \bigl(2 - 2\rho_w(\tau_d)\cos 2\pi f\tau_d\bigr)\,S_\mathrm{laser}(f) + S_\mathrm{noise}(f).$$

Segments much longer than the delay (`ρ_w ≈ 1`) resolve the fringes.  With
segments **no longer than the delay** (the default), `ρ_w = 0` exactly:
inside one segment `f(t)` and `f(t - τ_d)` are disjoint stretches of the
same process and their powers simply add, `E[S_beat] = 2·S_laser + S_noise`.
No notches, nothing to deconvolve, and `τ_d` only has to be known well
enough to choose the segment.  Here `τ_d·f_s = 61 250` samples, so the
default segment is 32 768 samples and this 4.2 ms record averages
K ≈ 250 of them.

**The fit.**  `DshFmPsd` fits the mean beat PSD with three physical terms,

$$M(f) = g(f)\,\bigl(S_\mathrm{white} + h_{-1}/f\bigr) + \sigma^2\,\tfrac{2 f_s}{\pi^2}\sin^2(\pi f/f_s),\qquad g(f) = 2 - 2\rho_w\cos 2\pi f\tau_d,$$

white FM (`S_white = Δν/π`, the Lorentzian linewidth), flicker FM
(`h₋₁/f`, reported as `res.flicker`) and the detection noise: additive
noise on the beat is a white angle error of `σ²` rad² per sample, whose
first difference has the `≈ f²` spectrum of the last term (reported as
`res.angle_noise_var`).  It bypasses the interferometer, so `g` does not shape it.

Each Welch bin is χ²-distributed around `M(f)`.  The coefficients maximize
the **Whittle likelihood** of the bins, `Σ [ln M + P/M]` minimized, whose
score has zero mean for any number of segments: the fit stays unbiased even
with one or two segments, where a median of the bins reads the skewed χ²
median and comes out low.  Bins far above the model (spurs) are screened
out with a χ² test.

In [ ]:
res_fm = analysis.estimate_linewidth(
    z_dsh, analysis.DshFmPsd(delay=TAU_D, f_shift=F_AOM), sampling_rate=FS
)

fig, ax = plotting.plot_frequency_noise_psd(
    res_fm, title="Laser FM-noise PSD (beat PSD / 2) with the fitted model"
)
ax.axhline(DNU_TRUE / np.pi, color="C3", ls="--", label=r"Truth $\Delta\nu/\pi$")
ax.legend()
plt.show()

print(
    f"fm_psd:    Δν = {res_fm.value / 1e3:.1f} kHz   (truth {DNU_TRUE / 1e3:.0f} kHz, "
    f"K = {res_fm.n_segments} segments, fit {res_fm.band[0] / 1e3:.0f} kHz - "
    f"{res_fm.band[1] / 1e6:.0f} MHz)"
)
print(
    f"           angle noise σ² = {res_fm.angle_noise_var:.2e} rad² "
    f"(N0/2 = {0.5 * 10 ** (-SNR_DB / 10):.2e} at {SNR_DB} dB)"
)

# Fringe-resolving segments give the same answer, from far fewer segments.
res_long = analysis.estimate_linewidth(
    z_dsh,
    analysis.DshFmPsd(delay=TAU_D, f_shift=F_AOM, nperseg=1 << 19),
    sampling_rate=FS,
)
print(
    f"nperseg 2^19 (resolves the fringes, K = {res_long.n_segments}): "
    f"Δν = {res_long.value / 1e3:.1f} kHz"
)

**Reading this plot**

* flat plateau at `Δν/π` -> white-FM noise, height *is* the linewidth; the
  red curve is the fitted model;
* a `1/f` rise at low `f` would be flicker FM (technical noise, current
  source, temperature, **spool acoustics**) - see [25](25-laser-linewidth-short-delay-and-flicker.ipynb); the fit carries it as
  its own term;
* the rise toward Nyquist is detection noise, the model's `σ²` term.

**Limitations to keep in mind**

* the model has three terms; anything else in the band (random-walk FM at
  the lowest bins, acoustic lines, colored receiver noise) biases it - where
  the PSD and the red curve disagree, fence the band with `f_min`/`f_max`
  and report the PSD;
* the noise term assumes a beat SNR well above the unwrap threshold;
* `τ_d` sets the default segment only.  If you pass segments longer than
  the delay (to resolve the fringes or lower frequencies), the fit depends
  on the notch positions and `τ_d` must be accurate: notch `k` moves by
  `k·Δτ/τ_d` of a lobe;
* everything the interferometer adds (fiber acoustics/thermal drift in the
  delay arm, AOM synthesizer phase noise) is charged to the laser.

## 5. Method 2 - increment-variance slope (AWGN-immune scalar)

For a Wiener laser phase, the variance of the lag-`a` increment of the
*measured* Δφ is linear in the lag (for `a ≤ τ_d`, so the two Wiener
increments do not overlap):

$$\operatorname{Var}[\Delta\varphi(t) - \Delta\varphi(t-a)] =
4\pi\,\Delta\nu\, a \;+\; 2\sigma_w^2 ,$$

The detection noise lands entirely in the **intercept**, so a straight-line
fit needs *no SNR knowledge* - the slope alone gives `Δν = slope/(4π)`.
Default lags auto-shrink for long delays (small lags give the variance
estimator many more independent averages).

In [ ]:
res_inc = analysis.estimate_linewidth(
    z_dsh, analysis.DshIncrement(delay=TAU_D, f_shift=F_AOM), sampling_rate=FS
)
print(f"increment: Δν = {res_inc.value / 1e3:.1f} kHz")

## 6. Method 3 - the textbook Lorentzian width

In the incoherent regime the beat line is Lorentzian with FWHM `2Δν` - the
classic recipe applies directly to the spectrum of section 2.  Two practical
subtleties:

* the **-3 dB width** is corrupted by 1/f noise, which builds a Gaussian
  core around the carrier (and keeps growing with observation time);
* the customary fix is to measure the width far down the wings - at
  `-20 dB`: for a Lorentzian, `W₋₂₀ = √99 · FWHM`, so
  `Δν = W₋₂₀ / (2√99)`.

`DshLorentzian` measures both and reports their ratio
as a lineshape diagnostic: `W₂₀/W₃ ≈ 9.95` for a Lorentzian, `≈ 2.6` for a
Gaussian - anything in between says "mixed", trust the deep width.
The result carries the beat spectrum, so the line with both width contours
plots directly.

In [ ]:
res_lor = analysis.estimate_linewidth(
    z_dsh, analysis.DshLorentzian(delay=TAU_D, nperseg=1 << 15), sampling_rate=FS
)
plotting.plot_dsh_beat_psd(res_lor)
print(
    f"lorentzian: Δν(-20 dB) = {res_lor.value / 1e3:.1f} kHz, "
    f"Δν(-3 dB) = {res_lor.linewidth_3db / 1e3:.1f} kHz, "
    f"W₂₀/W₃ = {res_lor.lineshape_ratio:.1f} (Lorentzian ≈ 9.95), "
    f"τ_d/τ_c = {res_lor.coherence_factor:.0f}"
)

## 7. Summary

| method | validity | needs | robust against |
|---|---|---|---|
| `DshFmPsd` | any regime | τ_d roughly (sets the segment) | regime, AWGN and flicker (fitted), spurs (screened), few segments |
| `DshIncrement` | any regime, white-FM laser | accurate τ_d | AWGN (intercept) |
| `DshLorentzian` | `τ_d/τ_c ≳ 6` only | resolution + ≳30 dB dynamic range | 1/f core (use deep width) |

Practical checklist for a real measurement
([27](27-measurement-template-self-heterodyne.ipynb) is the fill-in template):

1. **Plot the beat PSD first** - a smooth line => incoherent, all three
   methods apply; spike + fringes => coherent regime => forget the
   Lorentzian fit.
2. Choose `τ_d` for the *laser*: 100 kHz-class -> a 25-50 km spool is the
   standard incoherent setup; kHz-class -> decoherence is impractical, use a
   short delay + `DshFmPsd`/`DshIncrement` ([25](25-laser-linewidth-short-delay-and-flicker.ipynb)).
3. Sample fast enough that the beat (line + wings) fits inside `(0, f_s/2)`
   - the AOM must clear the beat half-bandwidth on both sides.  This
   constrains the *real* single-PD capture only: an IQ receiver (with or
   without the AOM) delivers complex samples with no such margin.  (No AOM
   at all?  See [26](26-laser-linewidth-self-homodyne-iq.ipynb).)
4. Check `dphi_var` against `2πΔν·τ_d` and the FM-PSD plateau against
   `Δν/π` - self-consistency across estimators is the best sanity check.
5. Compare the FM PSD with the fitted model (red curve): where they
   disagree the white + flicker + noise model is incomplete - fence the band
   (`f_min`/`f_max`) or report the PSD rather than one number.
6. Polarization fading ruins a single-polarization capture; with a
   dual-polarization receiver pass both outputs with
   `combine_polarizations=True` ([25](25-laser-linewidth-short-delay-and-flicker.ipynb)).
7. Everything the interferometer adds (fiber acoustics, AOM driver noise)
   is billed to the laser - measure the setup floor
   with a much better laser, or with a shorted (τ_d -> 0) interferometer.

In [ ]:
print("ground truth  :", f"{DNU_TRUE / 1e3:.0f} kHz")
print("fm_psd        :", f"{res_fm.value / 1e3:.1f} kHz")
print("increment     :", f"{res_inc.value / 1e3:.1f} kHz")
print("lorentzian    :", f"{res_lor.value / 1e3:.1f} kHz")

**Next:** [25 · Short delays, flicker noise and polarization fading](25-laser-linewidth-short-delay-and-flicker.ipynb).